<a href="https://colab.research.google.com/github/Shaifali07/langgraph_learning/blob/main/Prompt_chaining_sequential_Flow_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.0 MB/s eta 0:00:00


In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [10]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [24]:
class state(TypedDict):
  Job_role:str
  Skills:str
  Job_description:str

In [29]:
def generate_skills(s:state)->state:
  text=s['Job_role']
  prompt=ChatPromptTemplate.from_template("generate the 5-7 essential skills needed for the {text} role, generate the comma seperated list only. No other text.")
  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.3)
  output_parser = StrOutputParser()
  chain = prompt | llm | output_parser
  result = chain.invoke({"text": text})
  s['Skills']=result
  return (s)


In [30]:
def generate_job_description(s:state)->state:
  job_role=s['Job_role']
  skills=s['Skills']

  prompt = ChatPromptTemplate.from_template("generate the job description for the {job_role} role and the skills needed for that role are {skills}")
  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.3)
  output_parser = StrOutputParser()
  chain = prompt | llm | output_parser
  result = chain.invoke({"job_role":job_role, "skills":skills})
  s['Job_description']=result
  return (s)


In [31]:
graph=StateGraph(state)
graph.add_node('generate_skills',generate_skills)
graph.add_node('generate_job_description',generate_job_description)
graph.add_edge(START,'generate_skills')
graph.add_edge('generate_skills','generate_job_description')
graph.add_edge('generate_job_description',END)
workflow=graph.compile()
initial_state={"Job_role":'AI engineer'}
final_state=workflow.invoke(initial_state)
print(final_state)

{'Job_role': 'AI engineer', 'Skills': 'Programming skills in languages such as Python, Java, C++, data structures and algorithms, machine learning frameworks like TensorFlow or PyTorch, natural language processing, computer vision, cloud computing platforms like AWS or Azure.', 'Job_description': "**Job Title: AI Engineer**\n\n**Job Summary:**\n\nWe are seeking a highly skilled and experienced AI Engineer to join our team. As an AI Engineer, you will be responsible for designing, developing, and deploying artificial intelligence and machine learning models to drive business growth and innovation. You will work on a wide range of projects, from natural language processing and computer vision to predictive modeling and recommender systems. If you have a strong background in programming, data structures, and machine learning, and are passionate about building intelligent systems, we encourage you to apply.\n\n**Key Responsibilities:**\n\n* Design, develop, and deploy AI and machine learni

In [32]:
with open("graph.png", "wb") as f:
    f.write(workflow.get_graph().draw_mermaid_png())